# 16 - Evaluate external ECG-CXR bridge ablations

This notebook keeps three claims separate:

1. **Paired external supervision:** ECG waveform-machine diagnosis and CXR image-report retrieval.
2. **External ECG-CXR patient identity:** explicitly not evaluable because these separately sourced external datasets do not provide a shared patient/admission linkage.
3. **Concept-derived semantics:** cross-patient retrieval using explicit, assertion-aware clinical compatibility rules. This is a secondary, hypothesis-generating endpoint and is not a substitute for patient-identity validation.

All tests use held-out data and deterministic 10-, 100-, and 1000-candidate pools. In addition to Hit/Recall@K and MRR, the notebook reports multi-positive precision, recall, MAP, nDCG, AUROC, similarity margins, tie diagnostics for discrete codes, shuffled-query controls, per-relation semantic results, and codebook utilization/collapse measures.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
REPORT_ROOT = ARTIFACT_ROOT / "reports" / "ablation"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
CODE_ROOT = ARTIFACT_ROOT / "code"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_ablation
importlib.reload(ecg_cxr_ablation)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CANDIDATE_COUNTS = [10, 100, 1000]
MAX_QUERIES = int(os.environ.get("EXTERNAL_EVAL_MAX_QUERIES", "5000"))
if MAX_QUERIES < 1:
    raise ValueError("EXTERNAL_EVAL_MAX_QUERIES must be positive")

adapter_path = ARTIFACT_ROOT / "external" / "adapters" / "adapter_index.json"
adapter_index = json.loads(adapter_path.read_text())
if not isinstance(adapter_index, list) or not adapter_index:
    raise ValueError(f"No trained adapters in {adapter_path}; run notebook 15 first")

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

rows = {m: read_jsonl(ARTIFACT_ROOT / "external" / m / "manifest.jsonl")
        for m in ["ecg", "cxr"]}
concept_rows = {m: read_jsonl(ARTIFACT_ROOT / "external" / m / "concepts.jsonl")
                for m in ["ecg", "cxr"]}
concepts = {}
for modality in ["ecg", "cxr"]:
    if len(rows[modality]) != len(concept_rows[modality]):
        raise ValueError(f"{modality} manifest/concept length mismatch; rerun notebook 14")
    for index, (row, concept) in enumerate(zip(rows[modality], concept_rows[modality])):
        if str(row["sample_id"]) != str(concept.get("sample_id")):
            raise ValueError(f"{modality} concept order mismatch at row {index}")
    concepts[modality] = [item.get("concepts", []) for item in concept_rows[modality]]

print({"device": str(DEVICE), "adapter_runs": len(adapter_index),
       "rows": {m: len(v) for m, v in rows.items()},
       "report_root": str(REPORT_ROOT)})


In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def load_adapter(entry):
    state = torch_load(entry["checkpoint"])
    bridge_state = torch_load(state["bridge_checkpoint"])
    resolved = bridge_state["resolved_config"]
    bridge = ecg_cxr_ablation.AblationBridge(
        resolved["input_dims"], resolved["model_variant"],
        **resolved["model_hparams"]).to(DEVICE)
    bridge.load_state_dict(bridge_state["model"])
    from torch import nn

    class Adapter(nn.Module):
        def __init__(self):
            super().__init__()
            self.bridge = bridge
            dim = bridge.common_dim
            self.shared_text = state["shared_text"]
            self.residual = nn.ModuleDict({
                m: nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))
                for m in ["ecg", "cxr"]})
            names = ["shared"] if self.shared_text else ["ecg", "cxr"]
            self.text_projector = nn.ModuleDict({
                name: nn.Sequential(nn.LayerNorm(state["text_dim"]), nn.Linear(state["text_dim"], dim))
                for name in names})

        def quantize_with_codes(self, values):
            if not self.bridge.uses_vq:
                return F.normalize(values, dim=-1), None
            quantized, indices, _, _ = self.bridge.common_codebook(values)
            return F.normalize(quantized, dim=-1), indices

        def sensor(self, modality, values):
            return self.quantize_with_codes(values + self.residual[modality](values))

        def text(self, modality, values):
            key = "shared" if self.shared_text else modality
            return self.quantize_with_codes(self.text_projector[key](values))

    model = Adapter().to(DEVICE)
    model.load_state_dict(state["model"])
    return model.eval().requires_grad_(False), state, resolved

ENCODED = {}

@torch.inference_mode()
def encode_test(entry, modality):
    cache_key = (str(entry["checkpoint"]), modality)
    if cache_key in ENCODED:
        return ENCODED[cache_key]
    model, state, resolved = load_adapter(entry)
    experiment = state["experiment"]
    base = ARTIFACT_ROOT / "external" / modality
    continuous = np.load(
        base / "bridge" / experiment["variant"] / f"seed_{experiment['seed']}" / "continuous_common.npy",
        mmap_mode="r")
    text = np.load(base / f"text_{state['text_view']}_biomedclip.npy", mmap_mode="r")
    sensor_status = np.load(base / "encoding_status.npy", mmap_mode="r")
    text_status = np.load(base / f"text_{state['text_view']}_encoding_status.npy", mmap_mode="r")
    expected = len(rows[modality])
    lengths = [len(continuous), len(text), len(sensor_status), len(text_status)]
    if any(length != expected for length in lengths):
        raise ValueError(f"{modality} evaluation array/manifest mismatch: {lengths} vs {expected}")
    index = np.asarray([
        i for i, row in enumerate(rows[modality])
        if row["split"] == "test" and sensor_status[i] == 1 and text_status[i] == 1
    ], dtype="int64")
    if len(index) < 2:
        raise ValueError(f"Need at least two jointly encoded {modality} test rows; found {len(index)}")

    sensor_parts, text_parts, sensor_code_parts, text_code_parts = [], [], [], []
    for start in range(0, len(index), 1024):
        take = index[start:start + 1024]
        sensor, sensor_codes = model.sensor(
            modality, torch.from_numpy(np.array(continuous[take])).to(DEVICE))
        text_value, text_codes = model.text(
            modality, torch.from_numpy(np.array(text[take])).to(DEVICE))
        sensor_parts.append(sensor.cpu())
        text_parts.append(text_value.cpu())
        if sensor_codes is not None:
            sensor_code_parts.append(sensor_codes.cpu())
            text_code_parts.append(text_codes.cpu())
    result = {
        "index": index,
        "sensor": torch.cat(sensor_parts),
        "text": torch.cat(text_parts),
        "sensor_codes": torch.cat(sensor_code_parts).numpy() if sensor_code_parts else None,
        "text_codes": torch.cat(text_code_parts).numpy() if text_code_parts else None,
        "uses_vq": bool(model.bridge.uses_vq),
        "codebook_size": int(resolved["model_hparams"].get("common_codebook_size", 0)),
    }
    for key in ["sensor", "text"]:
        if not torch.isfinite(result[key]).all():
            raise ValueError(f"Non-finite {modality} {key} test embeddings")
    ENCODED[cache_key] = result
    return result


In [ ]:
def chance_hit(candidate_count, positive_count, k):
    n, p, k = int(candidate_count), int(positive_count), min(int(k), int(candidate_count))
    if p <= 0 or n <= 0:
        return np.nan
    if k > n - p:
        return 1.0
    no_positive = 1.0
    for offset in range(k):
        no_positive *= (n - p - offset) / (n - offset)
    return 1.0 - no_positive

def binary_ranking_metrics(scores, relevant, rng, ks=(1, 5, 10)):
    scores = np.asarray(scores, dtype="float64")
    relevant = np.asarray(relevant, dtype=bool)
    if scores.ndim != 1 or len(scores) != len(relevant) or not relevant.any():
        raise ValueError("Each candidate pool must contain at least one relevant item")
    # Random secondary keys give deterministic, unbiased tie breaking. This is
    # important for VQ models, where many candidates can have identical scores.
    order = np.lexsort((rng.random(len(scores)), -scores))
    ranked = relevant[order].astype("float64")
    positive_count = int(relevant.sum())
    first_rank = int(np.flatnonzero(ranked)[0]) + 1
    cumulative = np.cumsum(ranked)
    positive_scores, negative_scores = scores[relevant], scores[~relevant]
    best_positive = float(positive_scores.max())
    optimistic_rank = 1 + int((scores > best_positive).sum())
    tied_nonrelevant = int(np.isclose(negative_scores, best_positive, rtol=1e-6, atol=1e-8).sum())
    if len(negative_scores):
        ordered_negative = np.sort(negative_scores)
        wins = np.searchsorted(ordered_negative, positive_scores, side="left")
        ties = (np.searchsorted(ordered_negative, positive_scores, side="right") - wins)
        auc = float(np.mean((wins + 0.5 * ties) / len(negative_scores)))
        margin = best_positive - float(negative_scores.max())
        mean_negative = float(negative_scores.mean())
    else:
        auc, margin, mean_negative = np.nan, np.nan, np.nan
    result = {
        "reciprocal_rank": 1.0 / first_rank,
        "first_relevant_rank": float(first_rank),
        "optimistic_reciprocal_rank": 1.0 / optimistic_rank,
        "best_positive_tied_with_nonrelevant": float(tied_nonrelevant > 0),
        "nonrelevant_ties_at_best_positive": float(tied_nonrelevant),
        "auroc": auc,
        "similarity_margin": margin,
        "mean_positive_similarity": float(positive_scores.mean()),
        "mean_negative_similarity": mean_negative,
        "positive_prevalence": positive_count / len(scores),
    }
    for k in ks:
        take = min(k, len(scores))
        hits = float(cumulative[take - 1])
        precision = hits / take
        recall = hits / positive_count
        discounts = 1.0 / np.log2(np.arange(2, take + 2))
        dcg = float((ranked[:take] * discounts).sum())
        ideal = float(discounts[:min(positive_count, take)].sum())
        precision_curve = cumulative[:take] / np.arange(1, take + 1)
        ap = float((precision_curve * ranked[:take]).sum() / min(positive_count, take))
        result.update({
            f"hit_at_{k}": float(hits > 0),
            f"precision_at_{k}": precision,
            f"recall_at_{k}": recall,
            f"map_at_{k}": ap,
            f"ndcg_at_{k}": dcg / ideal if ideal else np.nan,
            f"chance_hit_at_{k}": chance_hit(len(scores), positive_count, take),
        })
    return result

def average_query_metrics(records):
    if not records:
        empty = {
            "queries": 0, "first_relevant_rank": None, "median_rank": None,
            "mrr": None, "optimistic_mrr": None,
            "best_positive_tied_with_nonrelevant": None,
            "nonrelevant_ties_at_best_positive": None, "auroc": None,
            "similarity_margin": None, "mean_positive_similarity": None,
            "mean_negative_similarity": None, "positive_prevalence": None,
        }
        for k in [1, 5, 10]:
            for name in ["hit", "precision", "recall", "map", "ndcg", "chance_hit"]:
                empty[f"{name}_at_{k}"] = None
        return empty
    keys = records[0].keys()
    result = {"queries": len(records)}
    for key in keys:
        values = np.asarray([item[key] for item in records], dtype="float64")
        result[key] = float(np.nanmean(values)) if not np.isnan(values).all() else None
    ranks = np.asarray([item["first_relevant_rank"] for item in records])
    result["median_rank"] = float(np.median(ranks))
    result["mrr"] = result.pop("reciprocal_rank")
    result["optimistic_mrr"] = result.pop("optimistic_reciprocal_rank")
    return result

def sampled_relevance(query, target, relevance, candidate_count, seed,
                      max_queries=MAX_QUERIES, excluded=None):
    rng = np.random.default_rng(seed)
    eligible = np.asarray(sorted(relevance), dtype="int64")
    if len(eligible) > max_queries:
        eligible = rng.choice(eligible, size=max_queries, replace=False)
    universe = np.arange(len(target), dtype="int64")
    records = []
    for query_index in eligible:
        positive = np.asarray(relevance[int(query_index)], dtype="int64")
        blocked = set() if excluded is None else set(excluded.get(int(query_index), ()))
        positive = np.asarray([value for value in positive if int(value) not in blocked], dtype="int64")
        if not len(positive):
            continue
        anchor = int(rng.choice(positive))
        allowed = np.asarray([value for value in universe
                              if int(value) != anchor and int(value) not in blocked], dtype="int64")
        sampled = rng.choice(
            allowed, size=min(candidate_count - 1, len(allowed)), replace=False)
        pool = np.concatenate([[anchor], sampled])
        relevant_pool = np.isin(pool, positive)
        scores = (target[pool] @ query[int(query_index)]).numpy()
        records.append(binary_ranking_metrics(scores, relevant_pool, rng))
    result = average_query_metrics(records)
    result.update({"requested_candidates": int(candidate_count),
                   "candidates": min(int(candidate_count), len(target))})
    return result

def normalized_group(modality, index):
    key = "machine_diagnosis" if modality == "ecg" else "report_text"
    return [" ".join(str(rows[modality][i][key]).lower().split()) for i in index]

def sampled_paired(query, candidate, groups, candidate_count, multipositive, seed=2026):
    n = len(query)
    if multipositive:
        lookup = {}
        for j, value in enumerate(groups):
            lookup.setdefault(value, []).append(j)
        relevance = {i: np.asarray(lookup[groups[i]], dtype="int64") for i in range(n)}
    else:
        relevance = {i: np.asarray([i], dtype="int64") for i in range(n)}
    return sampled_relevance(query, candidate, relevance, candidate_count, seed)


## Primary held-out Symile relational retrieval

This is the direct test of the codebook's intended ground truth. Within each official 10-candidate Symile group, the labeled ECG-CXR pair is the documented same-patient/encounter association. The metrics quantify retrieval of that association only; they do not assign a shared diagnosis to the two modalities. Unlike the earlier evaluation, discrete-score ties are broken deterministically and are also reported explicitly rather than counted optimistically as successful retrieval.


In [ ]:
@torch.inference_mode()
def encode_symile_bridge(experiment, split="test"):
    checkpoint = torch_load(experiment["checkpoint"])
    resolved = checkpoint["resolved_config"]
    model = ecg_cxr_ablation.AblationBridge(
        resolved["input_dims"], resolved["model_variant"],
        **resolved["model_hparams"]).to(DEVICE).eval()
    model.load_state_dict(checkpoint["model"])
    paths = {
        "ecg": ARTIFACT_ROOT / "features" / split / "ecg_hubert.npy",
        "cxr": ARTIFACT_ROOT / "features" / split / "cxr_raddino.npy",
        "manifest": ARTIFACT_ROOT / "manifests" / f"{split}.jsonl",
    }
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError("Run notebooks 02-04 before notebook 16; missing:\n" + "\n".join(missing))
    arrays = {m: np.load(paths[m], mmap_mode="r") for m in ["ecg", "cxr"]}
    manifest = read_jsonl(paths["manifest"])
    if any(len(arrays[m]) != len(manifest) for m in arrays):
        raise ValueError(f"Symile {split} feature/manifest row mismatch")
    output = {"ecg": [], "cxr": []}
    for start in range(0, len(manifest), 1024):
        stop = min(start + 1024, len(manifest))
        batch = {m: torch.from_numpy(np.array(arrays[m][start:stop])).float().to(DEVICE)
                 for m in ["ecg", "cxr"]}
        encoded = model(batch)
        for modality in output:
            output[modality].append(encoded[modality].common.cpu())
    output = {key: torch.cat(value) for key, value in output.items()}
    return output, manifest

def symile_group_retrieval(encoded, manifest, query_name, candidate_name, seed=5101):
    groups = {}
    for row_index, row in enumerate(manifest):
        if "label_hadm_id" not in row or "label" not in row:
            raise KeyError("Symile test manifest requires label_hadm_id and label")
        groups.setdefault(int(row["label_hadm_id"]), []).append(row_index)
    invalid = {key: value for key, value in groups.items()
               if len(value) != 10 or sum(int(manifest[i]["label"]) == 1 for i in value) != 1}
    if invalid:
        raise ValueError(f"Invalid Symile 10-candidate groups: {dict(list(invalid.items())[:10])}")
    rng = np.random.default_rng(seed)
    records = []
    for group_id in sorted(groups):
        pool = np.asarray(groups[group_id], dtype="int64")
        relevant = np.asarray([int(manifest[i]["label"]) == 1 for i in pool], dtype=bool)
        positive_index = int(pool[np.flatnonzero(relevant)[0]])
        scores = (encoded[candidate_name][pool] @ encoded[query_name][positive_index]).numpy()
        records.append(binary_ranking_metrics(scores, relevant, rng))
    result = average_query_metrics(records)
    result.update({"candidates": 10, "chance_hit_at_1_reference": 0.1,
                   "ground_truth": "official Symile same-patient/encounter pair; not semantic equivalence"})
    return result

bridge_experiments = {}
for entry in adapter_index:
    state = torch_load(entry["checkpoint"])
    experiment = state["experiment"]
    bridge_experiments[(experiment["variant"], int(experiment["seed"]))] = experiment

symile_relational_results = []
for (variant, seed), experiment in sorted(bridge_experiments.items()):
    encoded, manifest = encode_symile_bridge(experiment, "test")
    for direction, query_name, candidate_name in [
        ("ecg_to_cxr", "ecg", "cxr"), ("cxr_to_ecg", "cxr", "ecg")]:
        symile_relational_results.append({
            "bridge_variant": variant, "bridge_seed": seed, "direction": direction,
            **symile_group_retrieval(encoded, manifest, query_name, candidate_name)})
    print("Symile relational", variant, seed)

symile_relational_frame = pd.DataFrame(symile_relational_results)
symile_relational_path = REPORT_ROOT / "symile_relational_retrieval_detailed.csv"
symile_relational_frame.to_csv(symile_relational_path, index=False)
symile_relational_summary = symile_relational_frame.groupby(
    ["bridge_variant", "direction"], as_index=False).agg(
        seeds=("bridge_seed", "size"), hit_at_1_mean=("hit_at_1", "mean"),
        hit_at_1_std=("hit_at_1", "std"), hit_at_5_mean=("hit_at_5", "mean"),
        mrr_mean=("mrr", "mean"), map_at_10_mean=("map_at_10", "mean"),
        ndcg_at_10_mean=("ndcg_at_10", "mean"), auroc_mean=("auroc", "mean"),
        similarity_margin_mean=("similarity_margin", "mean"),
        tied_query_fraction_mean=("best_positive_tied_with_nonrelevant", "mean"))
symile_relational_summary_path = REPORT_ROOT / "symile_relational_retrieval_summary.csv"
symile_relational_summary.to_csv(symile_relational_summary_path, index=False)
display(symile_relational_summary.sort_values(["direction", "ndcg_at_10_mean"],
                                              ascending=[True, False]))
print("Saved:", symile_relational_path, symile_relational_summary_path)


In [ ]:
paired_results = []
for entry in adapter_index:
    encoded = {m: encode_test(entry, m) for m in ["ecg", "cxr"]}
    for modality in ["ecg", "cxr"]:
        item = encoded[modality]
        groups = normalized_group(modality, item["index"])
        for candidates in CANDIDATE_COUNTS:
            directions = [
                ("sensor_to_text", item["sensor"], item["text"]),
                ("text_to_sensor", item["text"], item["sensor"]),
            ]
            for direction, query, target in directions:
                for label_policy, multipositive in [
                    ("strict_instance", False), ("text_equivalent", True)]:
                    metrics = sampled_paired(
                        query, target, groups, candidates, multipositive=multipositive)
                    paired_results.append({**entry, "task": f"{modality}_{direction}",
                                           "label_policy": label_policy, **metrics})
    print("paired", entry["bridge_variant"], entry["semantic_variant"], entry["adapter_seed"])

paired_frame = pd.DataFrame(paired_results)
paired_path = REPORT_ROOT / "external_paired_retrieval.csv"
paired_frame.to_csv(paired_path, index=False)
metric_columns = [
    "hit_at_1", "hit_at_5", "hit_at_10", "recall_at_1", "recall_at_5", "recall_at_10",
    "precision_at_5", "precision_at_10", "map_at_10", "ndcg_at_10", "mrr", "auroc",
    "similarity_margin", "best_positive_tied_with_nonrelevant"]
aggregations = {f"{name}_mean": (name, "mean") for name in metric_columns}
aggregations.update({f"{name}_std": (name, "std") for name in ["hit_at_1", "mrr", "ndcg_at_10"]})
aggregations["seeds"] = ("adapter_seed", "size")
paired_summary = paired_frame.groupby(
    ["bridge_variant", "semantic_variant", "task", "label_policy", "candidates"],
    as_index=False).agg(**aggregations)
paired_summary_path = REPORT_ROOT / "external_paired_retrieval_summary.csv"
paired_summary.to_csv(paired_summary_path, index=False)
display(paired_summary.sort_values(["candidates", "task", "ndcg_at_10_mean"],
                                   ascending=[True, True, False]).head(40))
print("Saved:", paired_path, paired_summary_path)


## External ECG-CXR patient-identity validation: not evaluable

The external MIMIC-ECG and separately prepared MIMIC-CXR collections used here do not provide a verified shared patient/admission linkage. Consequently, external ECG-CXR patient-identity retrieval is not computed. The structured status files written below prevent an empty result from being mistaken for a failed run or a negative retrieval result.


In [ ]:
# These helpers remain available to the later semantic analysis so it can
# exclude matching identifiers if a future dataset supplies verified links.
# Their presence does not make identity validation evaluable in this cohort.
def identifier_values(modality, index, key):
    return [rows[modality][int(i)].get(key) for i in index]

def common_identity_key(ecg_item, cxr_item, keys=("hadm_id", "subject_id")):
    for key in keys:
        left = {value for value in identifier_values("ecg", ecg_item["index"], key)
                if value is not None}
        right = {value for value in identifier_values("cxr", cxr_item["index"], key)
                 if value is not None}
        if left & right:
            return key
    return None

EXTERNAL_IDENTITY_LINKAGE_VERIFIED = False
identity_status = {
    "status": "not_evaluable",
    "reason_code": "missing_shared_patient_identity",
    "reason": (
        "The external ECG and CXR datasets do not provide a verified shared patient "
        "or admission linkage; cross-sensor patient-identity retrieval cannot be evaluated."),
    "ecg_dataset": "MIMIC-IV-ECG waveform-machine-diagnosis pairs",
    "cxr_dataset": "separately prepared MIMIC-CXR image-report pairs",
    "metrics_computed": False,
    "interpretation": (
        "Do not interpret this endpoint as failed retrieval, zero performance, or external "
        "validation of the Symile patient-concurrence bridge."),
}
identity_path = REPORT_ROOT / "external_cross_sensor_identity.json"
identity_path.write_text(json.dumps(identity_status, indent=2))
identity_summary_path = REPORT_ROOT / "external_cross_sensor_identity_summary.csv"
identity_summary = pd.DataFrame([identity_status])
identity_summary.to_csv(identity_summary_path, index=False)
display(identity_summary)
print("Saved:", identity_path, identity_summary_path)


## Cross-patient semantic compatibility retrieval

This secondary analysis uses only concepts asserted as present. Negated and uncertain findings are excluded from positive labels. Same-admission or same-patient candidates are removed only when a verified cross-dataset identity linkage is available; no such linkage exists for the current external datasets. Unsupported@K is reported explicitly; it is the fraction of the top K without a preregistered compatible concept.


In [ ]:
COMPATIBILITY = {
    "ventricular_hypertrophy": {"cardiomegaly"},
    "paced_rhythm": {"support_device"},
    "heart_failure": {"cardiomegaly", "edema", "pleural_effusion"},
}

def assertion_sets(values):
    result = {"present": set(), "absent": set(), "uncertain": set()}
    for value in values:
        text = str(value).strip().lower()
        if ":" not in text:
            continue
        status, concept = text.split(":", 1)
        if status in result and concept:
            result[status].add(concept)
    return result

assertions = {
    modality: [assertion_sets(concepts[modality][int(i)])
               for i in encode_test(adapter_index[0], modality)["index"]]
    for modality in ["ecg", "cxr"]
}

def identity_exclusions(query_modality, query_item, target_modality, target_item):
    if not EXTERNAL_IDENTITY_LINKAGE_VERIFIED:
        return None, {}
    key = common_identity_key(
        query_item if query_modality == "ecg" else target_item,
        target_item if target_modality == "cxr" else query_item,
        keys=("subject_id", "hadm_id"))
    if key is None:
        return None, {}
    query_ids = identifier_values(query_modality, query_item["index"], key)
    target_ids = identifier_values(target_modality, target_item["index"], key)
    lookup = {}
    for j, value in enumerate(target_ids):
        if value is not None:
            lookup.setdefault(value, []).append(j)
    return key, {i: np.asarray(lookup[value], dtype="int64")
                 for i, value in enumerate(query_ids) if value in lookup}

def semantic_relevance(query_assertions, target_assertions, direction, exclusions):
    target_lookup = {}
    for j, values in enumerate(target_assertions):
        for concept in values["present"]:
            target_lookup.setdefault(concept, []).append(j)
    inverse = {}
    for source, targets in COMPATIBILITY.items():
        for target in targets:
            inverse.setdefault(target, set()).add(source)
    relevance = {}
    for i, values in enumerate(query_assertions):
        present = set(values["present"])
        desired = set(present)  # exact normalized concepts are semantic matches
        relation_sources = set()
        if direction == "ecg_to_cxr":
            for concept in present:
                mapped = COMPATIBILITY.get(concept, set())
                if mapped:
                    relation_sources.add(concept)
                    desired.update(mapped)
        else:
            for concept in present:
                mapped = inverse.get(concept, set())
                if mapped:
                    relation_sources.add(concept)
                    desired.update(mapped)
        relevant = {j for concept in desired for j in target_lookup.get(concept, [])}
        relevant.difference_update(set(exclusions.get(i, ())))
        if relevant:
            relevance[i] = np.asarray(sorted(relevant), dtype="int64")
    return relevance

semantic_results, relation_results = [], []
for entry in adapter_index:
    ecg_item, cxr_item = encode_test(entry, "ecg"), encode_test(entry, "cxr")
    item_assertions = {
        "ecg": [assertion_sets(concepts["ecg"][int(i)]) for i in ecg_item["index"]],
        "cxr": [assertion_sets(concepts["cxr"][int(i)]) for i in cxr_item["index"]],
    }
    directions = [
        ("ecg_to_cxr", "ecg", ecg_item, "cxr", cxr_item),
        ("cxr_to_ecg", "cxr", cxr_item, "ecg", ecg_item),
    ]
    for direction, query_modality, query_item, target_modality, target_item in directions:
        exclusion_key, exclusions = identity_exclusions(
            query_modality, query_item, target_modality, target_item)
        relevance = semantic_relevance(
            item_assertions[query_modality], item_assertions[target_modality], direction, exclusions)
        permutation = np.random.default_rng(4107).permutation(len(query_item["sensor"]))
        shuffled_query = query_item["sensor"][permutation]
        assertion_counts = {
            status: sum(bool(item[status]) for item in item_assertions[query_modality])
            for status in ["present", "absent", "uncertain"]}
        for candidates in CANDIDATE_COUNTS:
            metrics = sampled_relevance(
                query_item["sensor"], target_item["sensor"], relevance, candidates, seed=4101,
                excluded=exclusions)
            control = sampled_relevance(
                shuffled_query, target_item["sensor"], relevance, candidates, seed=4101,
                excluded=exclusions)
            semantic_results.append({
                **entry, "direction": direction,
                "ground_truth": "preregistered present-concept compatibility; not patient identity",
                "identity_exclusion_key": exclusion_key,
                "eligible_query_fraction": len(relevance) / len(query_item["sensor"]),
                **{f"queries_with_{key}_concepts": value for key, value in assertion_counts.items()},
                **metrics,
                "unsupported_rate_at_1": 1.0 - metrics.get("precision_at_1", np.nan),
                "unsupported_rate_at_5": 1.0 - metrics.get("precision_at_5", np.nan),
                "unsupported_rate_at_10": 1.0 - metrics.get("precision_at_10", np.nan),
                "shuffled_hit_at_1": control.get("hit_at_1"),
                "shuffled_mrr": control.get("mrr"),
                "shuffled_ndcg_at_10": control.get("ndcg_at_10"),
            })

        # Macro/per-relation analysis at the largest candidate pool.
        for source, targets in COMPATIBILITY.items():
            relation = f"{source}=>{'|'.join(sorted(targets))}"
            if direction == "ecg_to_cxr":
                query_required, target_required = {source}, set(targets)
            else:
                query_required, target_required = set(targets), {source}
            target_positions = np.asarray([
                j for j, values in enumerate(item_assertions[target_modality])
                if values["present"] & target_required], dtype="int64")
            relation_relevance = {}
            if len(target_positions):
                for i, values in enumerate(item_assertions[query_modality]):
                    if values["present"] & query_required:
                        allowed = np.asarray([j for j in target_positions
                                              if j not in set(exclusions.get(i, ()))], dtype="int64")
                        if len(allowed):
                            relation_relevance[i] = allowed
            relation_metrics = sampled_relevance(
                query_item["sensor"], target_item["sensor"], relation_relevance,
                max(CANDIDATE_COUNTS), seed=4201, excluded=exclusions)
            relation_results.append({**entry, "direction": direction, "relation": relation,
                                     "eligible_query_fraction": len(relation_relevance) / len(query_item["sensor"]),
                                     **relation_metrics})

semantic_path = REPORT_ROOT / "external_cross_sensor_semantic.json"
semantic_path.write_text(json.dumps(semantic_results, indent=2))
semantic_frame = pd.DataFrame(semantic_results)
semantic_summary_path = REPORT_ROOT / "external_cross_sensor_semantic_summary.csv"
semantic_summary = semantic_frame.groupby(
    ["bridge_variant", "semantic_variant", "direction", "candidates"], as_index=False).agg(
        seeds=("adapter_seed", "size"), queries_mean=("queries", "mean"),
        eligible_query_fraction_mean=("eligible_query_fraction", "mean"),
        hit_at_1_mean=("hit_at_1", "mean"), hit_at_1_std=("hit_at_1", "std"),
        hit_at_10_mean=("hit_at_10", "mean"), precision_at_10_mean=("precision_at_10", "mean"),
        recall_at_10_mean=("recall_at_10", "mean"), map_at_10_mean=("map_at_10", "mean"),
        ndcg_at_10_mean=("ndcg_at_10", "mean"), auroc_mean=("auroc", "mean"),
        mrr_mean=("mrr", "mean"), unsupported_rate_at_10_mean=("unsupported_rate_at_10", "mean"),
        shuffled_hit_at_1_mean=("shuffled_hit_at_1", "mean"))
semantic_summary.to_csv(semantic_summary_path, index=False)
relation_path = REPORT_ROOT / "external_cross_sensor_semantic_by_relation.csv"
pd.DataFrame(relation_results).to_csv(relation_path, index=False)
display(semantic_summary.sort_values(["candidates", "direction", "ndcg_at_10_mean"],
                                     ascending=[True, True, False]).head(40))
print("Saved:", semantic_path, semantic_summary_path, relation_path)


## Shared-codebook utilization and collapse diagnostics

Retrieval alone can hide codebook collapse. The following diagnostics report code utilization, effective perplexity, dominant-code share, complete code-sequence diversity, sensor-text code agreement, and the Jensen-Shannon divergence between external ECG and CXR sensor-code distributions. Continuous CLIP variants are retained with uses_vq=false and no code statistics.


In [ ]:
def code_usage_metrics(codes, codebook_size):
    codes = np.asarray(codes, dtype="int64")
    perplexities = []
    for position in range(codes.shape[1]):
        count = np.bincount(codes[:, position], minlength=codebook_size).astype("float64")
        probability = count[count > 0] / count.sum()
        perplexities.append(float(np.exp(-(probability * np.log(probability)).sum())))
    total = np.bincount(codes.reshape(-1), minlength=codebook_size).astype("float64")
    probability = total / total.sum()
    unique_sequences = len({tuple(value) for value in codes.tolist()})
    return {
        "rows": len(codes), "chunks": codes.shape[1],
        "used_codes": int((total > 0).sum()),
        "code_utilization": float((total > 0).mean()),
        "dead_code_fraction": float((total == 0).mean()),
        "top_code_share": float(probability.max()),
        "per_chunk_perplexity_mean": float(np.mean(perplexities)),
        "per_chunk_perplexity_min": float(np.min(perplexities)),
        "unique_sequence_fraction": unique_sequences / len(codes),
    }

def code_distribution_js(left, right, codebook_size):
    values = []
    for position in range(left.shape[1]):
        p = np.bincount(left[:, position], minlength=codebook_size).astype("float64")
        q = np.bincount(right[:, position], minlength=codebook_size).astype("float64")
        p, q = p / p.sum(), q / q.sum()
        middle = 0.5 * (p + q)
        left_term = np.where(p > 0, p * np.log((p + 1e-12) / (middle + 1e-12)), 0).sum()
        right_term = np.where(q > 0, q * np.log((q + 1e-12) / (middle + 1e-12)), 0).sum()
        values.append(0.5 * (left_term + right_term))
    return float(np.mean(values))

codebook_results = []
for entry in adapter_index:
    encoded = {m: encode_test(entry, m) for m in ["ecg", "cxr"]}
    if not encoded["ecg"]["uses_vq"]:
        codebook_results.append({**entry, "scope": "continuous_baseline", "uses_vq": False})
        continue
    size = encoded["ecg"]["codebook_size"]
    for modality in ["ecg", "cxr"]:
        item = encoded[modality]
        for representation in ["sensor", "text"]:
            code = item[f"{representation}_codes"]
            codebook_results.append({**entry, "scope": f"{modality}_{representation}",
                                     "uses_vq": True, **code_usage_metrics(code, size)})
        sensor_code, text_code = item["sensor_codes"], item["text_codes"]
        codebook_results.append({
            **entry, "scope": f"{modality}_paired_sensor_text", "uses_vq": True,
            "chunk_code_agreement": float((sensor_code == text_code).mean()),
            "full_sequence_agreement": float(np.all(sensor_code == text_code, axis=1).mean()),
        })
    codebook_results.append({
        **entry, "scope": "ecg_vs_cxr_sensor_distribution", "uses_vq": True,
        "mean_per_chunk_js_divergence": code_distribution_js(
            encoded["ecg"]["sensor_codes"], encoded["cxr"]["sensor_codes"], size),
    })

codebook_path = REPORT_ROOT / "external_codebook_diagnostics.csv"
pd.DataFrame(codebook_results).to_csv(codebook_path, index=False)
display(pd.DataFrame(codebook_results).head(40))
print("Saved:", codebook_path)


## Interpretation rules

- External ECG-CXR patient-identity validation is not evaluable because no verified shared patient/admission linkage is available.
- Do not report empty or not-evaluable identity outputs as zero performance, failed retrieval, or external validation.
- Symile same-patient retrieval remains the evidence-based patient-context endpoint and must not be called semantic or diagnostic agreement.
- Use paired ECG-diagnosis and CXR-report retrieval to show that external adapters preserve modality-specific meaning.
- Treat cross-patient concept compatibility, per-relation results, and unsupported@K as secondary semantic-transfer analyses.
- Never use absent or uncertain concepts as semantic positives; audit any LLM-derived concepts before evaluation.
- Prefer tie-broken MRR/nDCG to optimistic rank metrics for VQ models, and inspect utilization/perplexity before claiming codebook success.
- Compare prespecified variants using mean and standard deviation across seeds. Do not select a method using test performance.
